In [16]:
import pandas as pd
import time
import sys
import json
sys.path.insert(0, "../../utils/")
import xgboost as xgb
import lightgbm as lgb
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate
from sklearn.metrics import make_scorer, matthews_corrcoef
from joblib import dump
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score
)

In [17]:
REPR_NAMES = [
    #"embedding_antiviral_homology_90_esm1b_t33_650M_UR50S",
    #"embedding_antiviral_homology_90_esm2_t6_8M_UR50D",
    #"embedding_antiviral_homology_90_esm2_t12_35M_UR50D",
    #"embedding_antiviral_homology_90_esm2_t36_3B_UR50D",
    "embedding_antiviral_homology_90_protT5",
    #"frequency_antiviral_homology_90",
    #"one_hot_antiviral_homology_90",
]

In [18]:
MODELS = {
    "SVM": SVC,
    "KNN": KNeighborsClassifier,
    "LogisticRegression": LogisticRegression,
    "RandomForest": RandomForestClassifier,
    "AdaBoost": AdaBoostClassifier,
    "XGBoost": xgb.XGBClassifier,
    #"LGBM": lgb.LGBMClassifier,
    "GradientBoosting": GradientBoostingClassifier,
}

GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 5000],
        "min_samples_split": list(range(1, 51, 1)),
        "min_samples_leaf": list(range(1, 51, 1)),
        "max_depth": list(range(1, 51, 1)) + [None]
    },
}

In [19]:
GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 500, 1000, 3000],
        "min_samples_split": [2, 10, 20],
        "min_samples_leaf": [1, 4, 8],
        "max_features": ["sqrt", "log2"],
        "max_depth": [10, 30, None]
    },
    "AdaBoost": {
        "n_estimators": [50, 200, 500],
        "learning_rate": [0.01, 0.1, 1.0]
    },
    "GradientBoosting": {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0]
    },
    "XGBoost": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    "LGBM": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5, -1],
        "learning_rate": [0.01, 0.1],
        "num_leaves": [15, 31],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale"]
    },
    "LogisticRegression": {
        "C": [0.1, 1, 10],
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"]
    }
}

In [20]:
def get_model(model_name, seed):
    model_cls = MODELS[model_name]
    if model_name == "AdaBoost":
        model = model_cls(algorithm="SAMME")
    elif model_name == "KNN":
        model = model_cls(n_jobs=-1)
    elif model_name == "SVM":
        model = model_cls(probability=True, random_state=seed)
    elif model_name in ["RandomForest", "XGBoost", "LGBM"]:
        model = model_cls(random_state=seed, n_jobs=-1)
    else:
        model = model_cls(random_state=seed)
    return model

In [21]:
def undersampling(df_data, seed, repr_name):
    #ten_percent=df_data[df_data["target"]==2].sample(frac=0.10, random_state=seed)
    X=df_data.drop('target', axis=1)
    y=df_data['target']  
    #Se definen los objetos para submuestrear
    X['index']=X.index 
    undersampler=RandomUnderSampler(sampling_strategy='not minority', random_state=seed)    

    #Se aplica el submuestreo
    X_res, y_res=undersampler.fit_resample(X, y)
    df_resampled=pd.concat([X_res,y_res], axis=1)

    index_res=X_res['index']
    df_resampled.drop('index', axis=1, inplace=True)

    mask=~X['index'].isin(index_res)
    excluded_data=df_data[mask.values]
    #data_independent= pd.concat([ten_percent, excluded_data], axis=0)
    data_independent= pd.concat([excluded_data], axis=0)
    data_independent.reset_index(drop=True, inplace=True)
    data_independent.to_csv(f"../../data/numerical_rep/indep/data_independent_{repr_name}.csv", index=False)
    
    return df_resampled

In [22]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [23]:
def function_split(df_data, seed, repr_name):
    #Separa los datos
    data_under= undersampling(df_data, seed, repr_name)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [24]:
def metrics(model, predict_val, y_val, dataset, div, predict_proba=None):
    acc_value = accuracy_score(y_pred=predict_val, y_true=y_val) 
    recall_value = recall_score(y_pred=predict_val, y_true=y_val, average='weighted')
    precision_value = precision_score(y_pred=predict_val, y_true=y_val, average='weighted') 
    f1_value = f1_score(y_pred=predict_val, y_true=y_val, average='weighted')
    mcc_value = matthews_corrcoef(y_pred=predict_val, y_true=y_val)
    cm = confusion_matrix(y_pred=predict_val, y_true=y_val)
    cm_dict = pd.DataFrame(cm).to_dict()

    roc_auc_value = None
    if predict_proba is not None:
        try:
            roc_auc_value = roc_auc_score(y_val, predict_proba, multi_class='ovr', average='weighted')
        except Exception as e:
            print(f"Error computing ROC AUC: {e}")
            roc_auc_value = None

    df_metrics = pd.DataFrame([[dataset, model, div, acc_value, recall_value, precision_value, f1_value, mcc_value, roc_auc_value, cm_dict]],
                              columns=["dataset", "model", "sampling", "acc", "recall", "precision", "f1", "mcc", "roc_auc", "conf_matrix"])

    return df_metrics

In [25]:
def cross_function(model, X_train, y_train, cv):
    scoring_metrics = {
        "accuracy": "accuracy",
        "recall": "recall_weighted",
        "precision": "precision_weighted",
        "f1": "f1_weighted",
        "roc_auc_ovr": make_scorer(roc_auc_score, response_method="predict_proba", multi_class="ovr"),
        "mcc": make_scorer(matthews_corrcoef)
    }

    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics, return_train_score=True)

    df_val = pd.DataFrame({metric: scores[f'test_{metric}'] for metric in scoring_metrics})
    df_val['fit_time'] = scores['fit_time']
    df_val['score_time'] = scores['score_time']
    df_val['Dataset'] = 'Validation'

    df_train = pd.DataFrame({metric: scores[f'train_{metric}'] for metric in scoring_metrics})
    df_train['fit_time'] = scores['fit_time']
    df_train['score_time'] = scores['score_time']
    df_train['Dataset'] = 'Train'

    results_metrics = pd.concat([df_val, df_train], ignore_index=True)

    return results_metrics

In [ ]:
def function_train(model, model_name, repr_name, train, val, div, seed):
    # Separa los datos de entrenamiento y validación
    X_train = train.drop(columns="target").values
    y_train = train["target"].values

    # Combina los conjuntos de entrenamiento y validación para la validación cruzada
    train_all= pd.concat([train, val], ignore_index=True)
    target = train_all["target"]
    train_all.drop(columns="target", inplace=True)

    print(f"Train {model_name} with seed {seed} and division {div}")    
    results = []

    #Valdación cruzada antes de la búsqueda de hiperparámetros
    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed)
    cv_scores= cross_function(model, train_all, target, cv).copy()
    cv_scores["model"] = model_name
    cv_scores["sampling"] = div
    results.append(cv_scores)

    #Se entrena modelo solo para poder compararse con datos independientes
    model.fit(X_train, y_train)  
    dump(model, f"../../models_data/nobest_joblib/{repr_name}_{model_name}_{seed}_{div}.joblib")
    all_results = pd.concat(results, ignore_index=True)
    return all_results

In [27]:
def grid_function(model, model_name, repr_name, train, val, seed, div):
    grid = GridSearchCV(estimator=model, param_grid=GRIDS[model_name], cv=10, scoring="f1_weighted", n_jobs=-1)
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()
    print(f"GridSearchCV {model_name} with seed {seed} and division {div}")
    
    # Se realiza la búsqueda de hiperparámetros
    start_fit = time.time()
    grid.fit(X_train, y_train)
    elapsed_fit = time.time() - start_fit

    # Se obtienen los mejores parámetros y el mejor modelo
    best_model = grid.best_estimator_

    dump(best_model, f"../../models_data/best_joblib/{repr_name}_{model_name}_{seed}_{div}_best.joblib")

    # Se predice en el conjunto de entrenamiento y validación
    y_pred_train = best_model.predict(X_train)
    y_proba_train = best_model.predict_proba(X_train)

    y_pred_val = best_model.predict(X_val)
    y_proba_val = best_model.predict_proba(X_val)

    # Se guardan las métricas de entrenamiento y validación
    start_metrics = time.time()
    train_metrics = metrics(model_name, y_pred_train, y_train, "Train", div, predict_proba=y_proba_train)
    val_metrics = metrics(model_name, y_pred_val, y_val, "Validation", div, predict_proba=y_proba_val)
    elapsed_metrics = time.time() - start_metrics
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    results['fit_time'] = elapsed_fit
    results['metrics_time'] = elapsed_metrics
    return results

In [28]:
def main_train(df_data, seed, model_name, repr_name, grid_search=False):
    all_metrics = []
    all_metrics_grid = []
    df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = function_split(df_data, seed, repr_name)
    model=get_model(model_name, seed)
    all_metrics= pd.concat([function_train(model, model_name, repr_name, df_train, df_val, "Original", seed),
                            function_train(model, model_name, repr_name, df_train_under, df_val_under, "Under", seed),
                            function_train(model, model_name, repr_name, df_train_over, df_val_over, "Over", seed)], 
                            ignore_index=True)
    
    all_metrics.to_csv(f"../../models_data/metrics/{repr_name}_{model_name}_{seed}_metrics.csv", index=False)
    if grid_search:
        all_metrics_grid = pd.concat([grid_function(model, model_name, repr_name, df_train, df_val, seed, "Original"),
                                      grid_function(model, model_name, repr_name, df_train_under, df_val_under, seed, "Under"),
                                      grid_function(model, model_name, repr_name, df_train_over, df_val_over, seed, "Over")], 
                                      ignore_index=True)
        all_metrics_grid.to_csv(f"../../models_data/metrics/{repr_name}_{model_name}_{seed}_Grid_metrics.csv", index=False)    

In [29]:
seed= 42

In [ ]:
for repr_name in REPR_NAMES:
    print(f"Loading data for representation: {repr_name}")
    df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
    if "experimental_characteristics" in df_data.columns:
        df_data.drop(["experimental_characteristics"], axis=1, inplace=True)
    for model_name in MODELS.keys():
        print(f"Processing {model_name} with {repr_name}")
        start_time = time.time()
    
        main_train(df_data, seed, model_name, repr_name, grid_search=True)
    
        elapsed_time = time.time() - start_time
        print(f"Time taken for {model_name}: {elapsed_time:.2f} seconds")
        print(f"Finished {model_name}")
        print("=====================================")

Loading data for representation: embedding_antiviral_homology_90_protT5
Processing SVM with embedding_antiviral_homology_90_protT5
Train SVM with seed 42 and division Original
Train SVM with seed 42 and division Under
Train SVM with seed 42 and division Over
GridSearchCV SVM with seed 42 and division Original
GridSearchCV SVM with seed 42 and division Under
GridSearchCV SVM with seed 42 and division Over
Time taken for SVM: 107.33 seconds
Finished SVM
Processing KNN with embedding_antiviral_homology_90_protT5
Train KNN with seed 42 and division Original
Train KNN with seed 42 and division Under
Train KNN with seed 42 and division Over
GridSearchCV KNN with seed 42 and division Original
GridSearchCV KNN with seed 42 and division Under
GridSearchCV KNN with seed 42 and division Over
Time taken for KNN: 3.51 seconds
Finished KNN
Processing LogisticRegression with embedding_antiviral_homology_90_protT5
Train LogisticRegression with seed 42 and division Original
Train LogisticRegression wit

/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
